# RetainAI — Block 4: Tool-Using LLM Agent on Amazon Bedrock

The agent ties every earlier block together. For a customer (and optionally their message) it:

1. Pulls the profile and a **live churn score from the production SageMaker endpoint** (Block 2), with SHAP reasons (Block 1)
2. Classifies the customer's message with the **intent model** (Block 3)
3. Calls the **plan engine**: company policy plus the Block 5 bandit decide the resolution step, the offer, and
   whether to escalate, based on the customer's risk *and* their message. It can also search the **policy and FAQ (RAG)**
4. Writes the reply: resolve the issue first, then present the offer

Then we **evaluate** it like a production system: programmatic guardrail checks, LLM-as-judge quality scores,
latency, and token usage, comparing two models.

**Design principle:** code and learned models decide *what to do*; the LLM decides *how to say it*.
An earlier version let the LLM choose the offer, and testing showed it made tone-deaf choices (for example,
pushing a contract on a customer reporting a billing error). Moving decisions into the plan engine fixed that.

In [1]:
%pip install -q shap xgboost scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1. Setup

In [2]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, "src")
from agent import Toolbox, Agent, validate, judge, AGENT_MODEL, JUDGE_MODEL, CHALLENGER_MODEL, STRONG_MODEL

tools = Toolbox()
active = tools.customers[tools.customers["Churn"] == 0]
print(f"Loaded {len(tools.customers):,} customers | endpoint: {tools.endpoint} | policy sections: {len(tools.policy_chunks)}")
print("Agent:", AGENT_MODEL, "| Strong:", STRONG_MODEL, "| Challenger:", CHALLENGER_MODEL, "| Judge:", JUDGE_MODEL)

Loaded 7,043 customers | endpoint: retainai-churn-serverless | policy sections: 8
Agent: us.amazon.nova-2-lite-v1:0 | Strong: us.amazon.nova-pro-v1:0 | Challenger: us.amazon.nova-micro-v1:0 | Judge: us.meta.llama3-3-70b-instruct-v1:0


## 2. Test each tool directly (before any LLM is involved)

In [3]:
cid = active.sort_values("revenue_at_risk_12m", ascending=False).index[0]
show = lambda name, out: print(f"--- {name}\n{json.dumps(out, indent=2, default=str)}\n")

show("get_customer_profile", tools.get_customer_profile(cid))
show("score_churn_risk (live endpoint)", tools.score_churn_risk(cid))
show("classify_message", tools.classify_message("I want to cancel, the internet keeps dropping"))
show("recommend_offer (billing message)", tools.recommend_offer(cid, "My bill looks wrong this month."))
show("search_policy", tools.search_policy("where can I find my invoice"))

--- get_customer_profile
{
  "customer_id": "2081-VEYEH",
  "tenure_months": 3,
  "contract": "Month-to-month",
  "monthly_charges": 107.95,
  "internet_service": "Fiber optic",
  "payment_method": "Electronic check",
  "active_add_ons": [
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
  ],
  "segment": 4,
  "billing_anomaly_flag": false
}

--- score_churn_risk (live endpoint)
{
  "churn_probability": 0.6764,
  "risk_band": "high",
  "revenue_at_risk_12m": 876.19,
  "top_risk_drivers": [
    "is_month_to_month = 1",
    "tenure = 3",
    "InternetService = Fiber optic"
  ],
  "scored_by": "retainai-churn-serverless"
}

--- classify_message
{
  "intent": "delete_account",
  "intent_group": "leaving",
  "signals": {
    "leaving": true,
    "billing_problem": false,
    "complaint": false
  },
  "confidence": 0.488,
  "is_churn_signal": true
}

--- recommend_offer (billing message)
{
  "message_intent": {
    "intent": "check_invoi

## 3. Run the agent once and inspect its reasoning trace

In [4]:
agent = Agent(tools, model_id=AGENT_MODEL)
result = agent.run(cid, "My bill looks wrong this month.")

for step in result["trace"]:
    print(f"🔧 {step['tool']}({json.dumps(step['input'])})")
print(f"\nTurns: {result['turns']} | Latency: {result['latency_s']}s | Tokens: {result['usage']}\n")
print(json.dumps(result["output"], indent=2))
print("\nGuardrail checks:", validate(result, tools))

🔧 get_customer_profile({"customer_id": "2081-VEYEH"})
🔧 score_churn_risk({"customer_id": "2081-VEYEH"})
🔧 recommend_offer({"customer_id": "2081-VEYEH", "customer_message": "My bill looks wrong this month."})
🔧 search_policy({"query": "how to handle customer who says My bill looks wrong this month"})

Turns: 3 | Latency: 3.48s | Tokens: {'inputTokens': 6827, 'outputTokens': 358}

{
  "risk_level": "high",
  "key_reasons": [
    "is_month_to_month = 1",
    "tenure = 3",
    "InternetService = Fiber optic"
  ],
  "resolution": "Our billing team will review this month's charges and reply within one business day. If there is an error, the customer receives a credit of up to $25.",
  "offer_code": "CONTRACT_UPGRADE",
  "escalate_to_human": false,
  "customer_message": "I see your bill looks wrong this month, and you're on a month-to-month plan with us for just 3 months. Our billing team will review this month's charges and reply within one business day\u2014if there\u2019s an error, you\u20

## 4. Evaluation scenarios

A mix of risk levels and situations, including cases where the right answer is *not* to offer a discount.

In [5]:
by_risk = active.sort_values("churn_prob", ascending=False)
high = by_risk[by_risk["churn_prob"] >= 0.6].index
mid = by_risk[(by_risk["churn_prob"] >= 0.3) & (by_risk["churn_prob"] < 0.6)].index
low = by_risk[by_risk["churn_prob"] < 0.15].index
anomaly = active[(active["billing_anomaly"] == 1) & (active["churn_prob"] >= 0.3)].index
no_internet = active[(active["InternetService"] == "No") & (active["churn_prob"] >= 0.3)].index

def first(idx, n=0):
    return idx[n] if len(idx) > n else by_risk.index[n]

scenarios = [
    {"name": "High risk, proactive outreach", "customer_id": first(high, 1), "message": None},
    {"name": "High risk, price complaint", "customer_id": first(high, 2),
     "message": "My bill keeps going up and the service hasn't changed. Why am I paying this much?"},
    {"name": "High risk, wants to close account", "customer_id": first(high, 3),
     "message": "Third outage this month. I want to close my account."},
    {"name": "High risk, double charge", "customer_id": first(high, 4),
     "message": "You charged me twice this month and nobody has helped me."},
    {"name": "Medium risk, proactive", "customer_id": first(mid, 0), "message": None},
    {"name": "Medium risk, simple question", "customer_id": first(mid, 1),
     "message": "Where can I find my latest invoice?"},
    {"name": "Low risk, proactive (should be NO_OFFER)", "customer_id": first(low, 0), "message": None},
    {"name": "Low risk, simple question", "customer_id": first(low, 1),
     "message": "How do I update my payment method?"},
    {"name": "Billing anomaly flagged", "customer_id": first(anomaly, 0),
     "message": "My bill looks wrong this month."},
    {"name": "Low risk, says they're leaving (override)", "customer_id": first(low, 2),
     "message": "I'm switching providers next month, please cancel my service."},
    {"name": "Phone-only customer at risk", "customer_id": first(no_internet, 0), "message": None},
]
pd.DataFrame(scenarios)

,name,customer_id,message
0,"High risk, proactive outreach",1941-HOSAM,None
1,"High risk, price complaint",0021-IKXGC,My bill keeps going up and the service hasn't ...
2,"High risk, wants to close account",4912-PIGUY,Third outage this month. I want to close my ac...
3,"High risk, double charge",7577-SWIFR,You charged me twice this month and nobody has...
4,"Medium risk, proactive",0480-BIXDE,None
5,"Medium risk, simple question",7729-JTEEC,Where can I find my latest invoice?
6,"Low risk, proactive (should be NO_OFFER)",4797-MIWUM,None
7,"Low risk, simple question",9060-HJJRW,How do I update my payment method?
8,Billing anomaly flagged,0771-WLCLA,My bill looks wrong this month.
9,"Low risk, says they're leaving (override)",4332-MUOEZ,"I'm switching providers next month, please can..."


## 5. Evaluate the agent model

In [6]:
def evaluate(model_id, scenarios):
    runner, rows, runs = Agent(tools, model_id=model_id), [], []
    for s in scenarios:
        res = runner.run(s["customer_id"], s["message"])
        checks = validate(res, tools)
        scores = judge(res, tools) if res["output"] else {}
        out = res["output"] or {}
        rows.append({"scenario": s["name"], "resolution": out.get("resolution"), "offer": out.get("offer_code"),
                     "escalate": out.get("escalate_to_human"),
                     **{k: v for k, v in checks.items()}, "judge_avg": scores.get("judge_avg"),
                     "judge_comment": scores.get("comment"), "latency_s": res["latency_s"],
                     "input_tokens": res["usage"]["inputTokens"], "output_tokens": res["usage"]["outputTokens"],
                     "tool_calls": len(res["trace"])})
        runs.append(res)
        print(f"{'✅' if checks['compliant'] else '❌'} {s['name']:<42} {out.get('resolution')} + {out.get('offer_code')}  judge={scores.get('judge_avg')}")
    return pd.DataFrame(rows), runs

agent_df, agent_runs = evaluate(AGENT_MODEL, scenarios)
agent_df.drop(columns=["judge_comment"])

✅ High risk, proactive outreach              PROACTIVE_CHECK_IN + CONTRACT_UPGRADE  judge=3.75
✅ High risk, price complaint                 Our billing team will review this month's charges and reply within one business day. If there is an error, you will receive a credit of up to $25. + CONTRACT_UPGRADE  judge=4.5
✅ High risk, wants to close account          A retention specialist will contact the customer within one business day. + CONTRACT_UPGRADE  judge=4.5
✅ High risk, double charge                   Our billing team will review this month's charges and reply within one business day. If there is an error, you will receive a credit of up to $25. + CONTRACT_UPGRADE  judge=4.5
❌ Medium risk, proactive                     PROACTIVE_CHECK_IN + CONTRACT_UPGRADE  judge=3.75
✅ Medium risk, simple question               Answer the question directly using the Customer FAQ from search_policy. + NO_OFFER  judge=4.25
✅ Low risk, proactive (should be NO_OFFER)   Thank you for being with us for 

,scenario,resolution,offer,escalate,valid_json,called_required_tools,offer_is_eligible,follows_plan,resolution_mentioned,discount_within_policy,...,no_internal_terms,no_invented_perks,escalation_correct,under_120_words,compliant,judge_avg,latency_s,input_tokens,output_tokens,tool_calls
0,"High risk, proactive outreach",PROACTIVE_CHECK_IN,CONTRACT_UPGRADE,False,True,True,True,True,True,True,...,True,True,True,True,True,3.75,1.60,3861,240,3
1,"High risk, price complaint",Our billing team will review this month's char...,CONTRACT_UPGRADE,False,True,True,True,True,True,True,...,True,True,True,True,True,4.50,2.85,6650,353,4
2,"High risk, wants to close account",A retention specialist will contact the custom...,CONTRACT_UPGRADE,True,True,True,True,True,True,True,...,True,True,True,True,True,4.50,1.72,3928,280,3
3,"High risk, double charge",Our billing team will review this month's char...,CONTRACT_UPGRADE,True,True,True,True,True,True,True,...,True,True,True,True,True,4.50,2.51,6748,334,4
4,"Medium risk, proactive",PROACTIVE_CHECK_IN,CONTRACT_UPGRADE,False,True,True,True,True,True,True,...,True,False,True,True,False,3.75,1.73,3882,238,3
5,"Medium risk, simple question",Answer the question directly using the Custome...,NO_OFFER,False,True,True,True,True,True,True,...,True,True,True,True,True,4.25,2.18,4136,266,4
6,"Low risk, proactive (should be NO_OFFER)",Thank you for being with us for 7 months. How ...,NO_OFFER,False,True,True,True,True,True,True,...,True,True,True,True,True,4.00,1.82,3602,232,3
7,"Low risk, simple question",Answer the question directly using the Custome...,NO_OFFER,False,True,True,True,True,True,True,...,True,True,True,True,True,4.00,1.65,3903,248,4
8,Billing anomaly flagged,Our billing team will review this month's char...,CONTRACT_UPGRADE,False,True,True,True,True,True,True,...,True,True,True,True,True,4.00,2.56,6752,325,4
9,"Low risk, says they're leaving (override)",A retention specialist will contact the custom...,LOYALTY_DISCOUNT,True,True,True,True,True,True,True,...,True,True,True,True,True,4.75,1.87,3824,258,3


In [7]:
# Read a few of the actual messages the agent wrote
for res in agent_runs[:4]:
    out = res["output"] or {}
    print(f"[{res['customer_id']}] {res['message'] or '(proactive)'}")
    print(f"  resolution: {out.get('resolution')} | offer: {out.get('offer_code')} | escalate: {out.get('escalate_to_human')}")
    print(f"  -> {out.get('customer_message')}\n")

[1941-HOSAM] (proactive)
  resolution: PROACTIVE_CHECK_IN | offer: CONTRACT_UPGRADE | escalate: False
  -> Thanks for being with us for 1 months. How can we help make your service better? If you’d like to switch to a 1-year contract, you could save $9.01 per month (about $108 over the year), bringing your monthly bill from $90.10 to $81.09.

[0021-IKXGC] My bill keeps going up and the service hasn't changed. Why am I paying this much?
  resolution: Our billing team will review this month's charges and reply within one business day. If there is an error, you will receive a credit of up to $25. | offer: CONTRACT_UPGRADE | escalate: False
  -> I see you’ve been with us just one month on a month-to-month plan, and your current bill is $72.10. We’ll have our billing team check this for you right away. If we find an error, you’ll get a credit of up to $25. To help keep costs steady, you might also consider switching to a 1-year contract, which would drop your monthly bill to $64.89—that’s a 

## 6. Model comparison: does a bigger or smaller model change the outcome?

In [8]:
strong_df, strong_runs = evaluate(STRONG_MODEL, scenarios)
challenger_df, challenger_runs = evaluate(CHALLENGER_MODEL, scenarios)

def summarize(df, model):
    return {"model": model.split(".")[-1], "compliance_rate": df["compliant"].mean(),
            "avg_judge_score": df["judge_avg"].mean(), "avg_latency_s": df["latency_s"].mean(),
            "avg_input_tokens": df["input_tokens"].mean(), "avg_output_tokens": df["output_tokens"].mean(),
            "avg_tool_calls": df["tool_calls"].mean()}

comparison = pd.DataFrame([summarize(agent_df, AGENT_MODEL), summarize(strong_df, STRONG_MODEL),
                           summarize(challenger_df, CHALLENGER_MODEL)])
comparison.round(3)

✅ High risk, proactive outreach              PROACTIVE_CHECK_IN + CONTRACT_UPGRADE  judge=3.5
✅ High risk, price complaint                 Our billing team will review this month's charges and reply within one business day. If there is an error, the customer receives a credit of up to $25. + CONTRACT_UPGRADE  judge=4.0
✅ High risk, wants to close account          ESCALATE_TO_SPECIALIST + CONTRACT_UPGRADE  judge=4.5
✅ High risk, double charge                   Our billing team will review this month's charges and reply within one business day. If there is an error, the customer receives a credit of up to $25. + CONTRACT_UPGRADE  judge=4.25
✅ Medium risk, proactive                     Thank you for being with us for 19 months. We're checking in to ensure everything is working well. Consider switching to a 1-year contract with 10% off the monthly charge, saving $9.68 per month (about $116 over the year), bringing the monthly bill from $96.80 to $87.12. + CONTRACT_UPGRADE  judge=3.75
✅ Med

,model,compliance_rate,avg_judge_score,avg_latency_s,avg_input_tokens,avg_output_tokens,avg_tool_calls
0,nova-2-lite-v1:0,0.818,4.136,2.043,4642.727,276.909,3.455
1,nova-pro-v1:0,1.000,3.955,3.659,3752.455,357.364,3.273
2,nova-micro-v1:0,0.909,3.955,2.665,4026.455,509.545,3.273


In [9]:
# Which guardrails failed, per model?
check_cols = ["valid_json", "called_required_tools", "offer_is_eligible", "follows_plan", "resolution_mentioned",
              "discount_within_policy", "no_internal_terms", "no_invented_perks", "escalation_correct", "under_120_words"]
pd.DataFrame({AGENT_MODEL.split(".")[-1]: agent_df[check_cols].mean(),
              STRONG_MODEL.split(".")[-1]: strong_df[check_cols].mean(),
              CHALLENGER_MODEL.split(".")[-1]: challenger_df[check_cols].mean()}).round(2)

,nova-2-lite-v1:0,nova-pro-v1:0,nova-micro-v1:0
valid_json,1.00,1.0,1.00
called_required_tools,1.00,1.0,0.91
offer_is_eligible,1.00,1.0,1.00
follows_plan,1.00,1.0,1.00
resolution_mentioned,1.00,1.0,1.00
discount_within_policy,1.00,1.0,1.00
no_internal_terms,1.00,1.0,1.00
no_invented_perks,0.82,1.0,1.00
escalation_correct,1.00,1.0,1.00
under_120_words,1.00,1.0,1.00


## 7. Save results and example transcripts

In [10]:
os.makedirs("artifacts", exist_ok=True); os.makedirs("docs", exist_ok=True)
agent_df.assign(model=AGENT_MODEL).to_csv("artifacts/agent_eval.csv", index=False)
challenger_df.assign(model=CHALLENGER_MODEL).to_csv("artifacts/agent_eval_challenger.csv", index=False)
strong_df.assign(model=STRONG_MODEL).to_csv("artifacts/agent_eval_strong.csv", index=False)
json.dump(comparison.round(4).to_dict(orient="records"), open("artifacts/agent_comparison.json", "w"), indent=2)

lines = ["# RetainAI Agent — Example Transcripts\n", f"Model: `{AGENT_MODEL}`\n"]
for s, res in zip(scenarios, agent_runs):
    out = res["output"] or {}
    lines += [f"## {s['name']}\n", f"**Customer message:** {s['message'] or '_(proactive outreach)_'}\n",
              "**Tool calls:** " + " → ".join(t["tool"] for t in res["trace"]) + "\n",
              f"**Resolution:** `{out.get('resolution')}` | **Offer:** `{out.get('offer_code')}` | **Escalate:** {out.get('escalate_to_human')}\n",
              f"**Internal reasons:** {', '.join(out.get('key_reasons', []))}\n",
              f"> {out.get('customer_message')}\n"]
open("docs/agent_examples.md", "w").write("\n".join(lines))
print(comparison.round(3).to_string(index=False))

           model  compliance_rate  avg_judge_score  avg_latency_s  avg_input_tokens  avg_output_tokens  avg_tool_calls
nova-2-lite-v1:0            0.818            4.136          2.043          4642.727            276.909           3.455
   nova-pro-v1:0            1.000            3.955          3.659          3752.455            357.364           3.273
 nova-micro-v1:0            0.909            3.955          2.665          4026.455            509.545           3.273
